# 💧 Panchnama — Water Model Fine-Tuning Pipeline (Kaggle)

This notebook fine-tunes the computer-vision models for the **Water Sector**:
1. **Water Body Detector** (YOLOv8n Object Detection for lakes/rivers/water bodies)
2. **Water Change Detector** (PyTorch Siamese Change Detection Network for hydrological monitoring)

All models export to `.onnx` format ready for deployment in `apps/ml-service/weights/`.


## 1. GPU Check & Environment Setup

In [ ]:
!pip install -q ultralytics onnx onnxscript onnxsim onnxruntime albumentations opencv-python-headless pyyaml tqdm requests

import os
import sys
import json
import shutil
import random
import numpy as np
import torch
import torch.nn as nn
from pathlib import Path
from dataclasses import dataclass

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"⚡ Device: {device}")
if device == "cuda":
    print(f"🎮 GPU: {torch.cuda.get_device_name(0)}")
    print(f"💾 VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("⚠️ Warning: CUDA GPU not detected. Running on CPU.")


## 2. Water Pipeline Configuration

In [ ]:
@dataclass
class WaterConfig:
    base_dir: Path = Path("./water_training")
    data_dir: Path = base_dir / "datasets"
    runs_dir: Path = base_dir / "runs"
    output_dir: Path = base_dir / "exported_weights"

    # Water Body Detector (YOLOv8n)
    water_epochs: int = 15
    water_imgsz: int = 640
    water_batch: int = 16
    water_onnx_name: str = "water_yolov8n.onnx"

    # Water Change Detector (Siamese ChangeFormer)
    change_epochs: int = 20
    change_imgsz: int = 256
    change_batch: int = 8
    change_lr: float = 1e-3
    change_onnx_name: str = "changeformer_water.onnx"

    def __post_init__(self):
        self.data_dir.mkdir(parents=True, exist_ok=True)
        self.runs_dir.mkdir(parents=True, exist_ok=True)
        self.output_dir.mkdir(parents=True, exist_ok=True)

cfg = WaterConfig()
print("✓ Water Config initialized.")
print(f"📁 Output path: {cfg.output_dir.resolve()}")


## 3. Water Datasets Preparation

In [ ]:
import cv2

def prepare_water_datasets(data_dir: Path):
    # 1. Water Body Detection Dataset
    water_dir = data_dir / "water_bodies"
    for split in ["train", "val"]:
        (water_dir / split / "images").mkdir(parents=True, exist_ok=True)
        (water_dir / split / "labels").mkdir(parents=True, exist_ok=True)
        num_samples = 40 if split == "train" else 10
        for i in range(num_samples):
            img = np.random.randint(100, 220, (640, 640, 3), dtype=np.uint8)
            # Add blue water bodies
            num_lakes = random.randint(1, 4)
            labels = []
            for _ in range(num_lakes):
                cx, cy = random.randint(80, 560), random.randint(80, 560)
                r = random.randint(30, 70)
                cv2.circle(img, (cx, cy), r, (random.randint(180, 240), random.randint(100, 150), 20), -1)
                w, h = (r * 2) / 640.0, (r * 2) / 640.0
                labels.append(f"0 {cx/640.0:.4f} {cy/640.0:.4f} {w:.4f} {h:.4f}")

            cv2.imwrite(str(water_dir / split / "images" / f"water_{i:03d}.jpg"), img)
            with open(water_dir / split / "labels" / f"water_{i:03d}.txt", "w") as f:
                f.write("\n".join(labels))

    yaml_str = f"path: {water_dir.resolve()}\ntrain: train/images\nval: val/images\nnc: 1\nnames: ['water']\n"
    with open(water_dir / "dataset.yaml", "w") as f:
        f.write(yaml_str)

    # 2. Water Change Dataset (Hydrological flooding / dryup pairs)
    change_dir = data_dir / "water_change"
    for split in ["train", "val"]:
        (change_dir / split / "A").mkdir(parents=True, exist_ok=True)
        (change_dir / split / "B").mkdir(parents=True, exist_ok=True)
        (change_dir / split / "label").mkdir(parents=True, exist_ok=True)
        num_samples = 30 if split == "train" else 10
        for i in range(num_samples):
            img_a = np.random.randint(80, 200, (256, 256, 3), dtype=np.uint8)
            img_b = img_a.copy()
            mask = np.zeros((256, 256), dtype=np.uint8)
            if random.random() > 0.3:
                x1, y1 = random.randint(20, 150), random.randint(20, 150)
                x2, y2 = x1 + random.randint(50, 80), y1 + random.randint(50, 80)
                # Blue water shift
                img_b[y1:y2, x1:x2] = np.array([220, 120, 30], dtype=np.uint8)
                mask[y1:y2, x1:x2] = 255

            cv2.imwrite(str(change_dir / split / "A" / f"pair_{i:03d}.png"), img_a)
            cv2.imwrite(str(change_dir / split / "B" / f"pair_{i:03d}.png"), img_b)
            cv2.imwrite(str(change_dir / split / "label" / f"pair_{i:03d}.png"), mask)

    print("✓ Water datasets prepared.")
    return water_dir / "dataset.yaml", change_dir

water_yaml, change_dir = prepare_water_datasets(cfg.data_dir)


## 4. Train Water Body Detector (YOLOv8n)

In [ ]:
from ultralytics import YOLO

print("💧 Fine-tuning YOLOv8n Water Body Detector...")
water_model = YOLO("yolov8n.pt")
water_model.train(
    data=str(water_yaml),
    epochs=cfg.water_epochs,
    imgsz=cfg.water_imgsz,
    batch=cfg.water_batch,
    project=str(cfg.runs_dir),
    name="water_detector",
    exist_ok=True,
    verbose=False
)

exported_onnx = water_model.export(format="onnx", opset=12, dynamic=True)
shutil.move(str(exported_onnx), cfg.output_dir / cfg.water_onnx_name)
print(f"✅ Exported: {cfg.output_dir / cfg.water_onnx_name}")


## 5. Train Water Change Detector (Siamese Net)

In [ ]:
from torch.utils.data import Dataset, DataLoader

class WaterChangeDataset(Dataset):
    def __init__(self, root: Path, split: str = "train", imgsz: int = 256):
        self.root = root / split
        self.imgsz = imgsz
        self.a_paths = sorted(list((self.root / "A").glob("*.png")))

    def __len__(self):
        return len(self.a_paths)

    def __getitem__(self, idx):
        a_path = self.a_paths[idx]
        b_path = self.root / "B" / a_path.name
        lbl_path = self.root / "label" / a_path.name

        img_a = cv2.imread(str(a_path))
        img_b = cv2.imread(str(b_path))
        lbl = cv2.imread(str(lbl_path), cv2.IMREAD_GRAYSCALE)

        img_a = cv2.resize(img_a, (self.imgsz, self.imgsz)).astype(np.float32) / 255.0
        img_b = cv2.resize(img_b, (self.imgsz, self.imgsz)).astype(np.float32) / 255.0
        lbl = (cv2.resize(lbl, (self.imgsz, self.imgsz)) > 128).astype(np.float32)

        return torch.from_numpy(img_a).permute(2, 0, 1), torch.from_numpy(img_b).permute(2, 0, 1), torch.from_numpy(lbl).unsqueeze(0)

class WaterChangeDetector(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True)
        )
        self.decoder = nn.Sequential(
            nn.Conv2d(64, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 1, 1)
        )

    def forward(self, img_a, img_b):
        fa = self.encoder(img_a)
        fb = self.encoder(img_b)
        diff = torch.abs(fa - fb)
        return self.decoder(diff)

print("🔄 Training Water Change Detector...")
train_loader = DataLoader(WaterChangeDataset(change_dir, "train", cfg.change_imgsz), batch_size=cfg.change_batch, shuffle=True)
model_water_cf = WaterChangeDetector().to(device)
opt = torch.optim.AdamW(model_water_cf.parameters(), lr=cfg.change_lr)
criterion = nn.BCEWithLogitsLoss()

model_water_cf.train()
for epoch in range(cfg.change_epochs):
    loss_val = 0.0
    for img_a, img_b, lbl in train_loader:
        img_a, img_b, lbl = img_a.to(device), img_b.to(device), lbl.to(device)
        opt.zero_grad()
        loss = criterion(model_water_cf(img_a, img_b), lbl)
        loss.backward()
        opt.step()
        loss_val += loss.item()
    if (epoch + 1) % 5 == 0:
        print(f"   Epoch [{epoch+1}/{cfg.change_epochs}] - Loss: {loss_val/len(train_loader):.4f}")

# ONNX Export
model_water_cf.eval().to("cpu")
dummy_a = torch.randn(1, 3, cfg.change_imgsz, cfg.change_imgsz)
dummy_b = torch.randn(1, 3, cfg.change_imgsz, cfg.change_imgsz)
torch.onnx.export(
    model_water_cf, (dummy_a, dummy_b), str(cfg.output_dir / cfg.change_onnx_name),
    opset_version=12, input_names=["before", "after"], output_names=["change_logits"],
    dynamic_axes={"before": {0: "batch"}, "after": {0: "batch"}, "change_logits": {0: "batch"}}
)
print(f"✅ Exported: {cfg.output_dir / cfg.change_onnx_name}")


## 6. Export Summary & Files Download

In [ ]:
import onnxruntime as ort

print("🎉 WATER TRAINING COMPLETE!")
print("\n📁 Generated ONNX Files in `water_training/exported_weights/`:")
for f in [cfg.water_onnx_name, cfg.change_onnx_name]:
    p = cfg.output_dir / f
    session = ort.InferenceSession(str(p), providers=["CPUExecutionProvider"])
    inputs = [i.name for i in session.get_inputs()]
    outputs = [o.name for o in session.get_outputs()]
    print(f"  - {f:<25} ({p.stat().st_size / 1e6:.2f} MB) | Inputs: {inputs} | Outputs: {outputs}")

print("\n⬇️ Copy these 2 files into your local repo under `apps/ml-service/weights/`:")
print(f"  1. {cfg.water_onnx_name}")
print(f"  2. {cfg.change_onnx_name}")
